In [1]:
import os
import scanpy as sc

In [5]:
import scvi

/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/.venv/lib/python3.12/site-packages/pyro/ops/stats.py:527: SyntaxWarning: invalid escape sequence '\g'
  we have :math:`ES^{*}(P,Q) \ge ES^{*}(Q,Q)` with equality holding if and only if :math:`P=Q`, i.e.


In [6]:
scvi.__version__

'1.4.3'

In [7]:
print(scvi.__file__)
print(scvi.__version__ if hasattr(scvi, "__version__") else "no version")

/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/.venv/lib/python3.12/site-packages/scvi/__init__.py
1.4.3


In [8]:
adata_path = os.path.join("data", "lung_atlas_preprocessed.h5ad")
adata_path
print(os.getcwd())

/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/src/PeakVI


In [17]:
# load data from root folder
adata_path = "/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/data/lung_atlas_preprocessed.h5ad"
adata = sc.read(
    adata_path,
    backup_url="https://figshare.com/ndownloader/files/52859288",
)
adata

AnnData object with n_obs × n_vars = 4585 × 33142
    obs: 'batch_id'
    var: 'chr', 'start', 'end', 'n_cells'

In [20]:
scvi.model.PEAKVI.setup_anndata(adata)
model = scvi.model.PEAKVI(adata)
model.train(accelerator='cpu') # use accelerator='cuda' if you have a GPU available, 'mps' on macOS or 'cpu' otherwise

# Retrieve the latent representation and store it in adata.obsm

PEAKVI_LATENT_KEY = "X_peakvi"

latent = model.get_latent_representation()
adata.obsm[PEAKVI_LATENT_KEY] = latent
latent.shape

PEAKVI_CLUSTERS_KEY = "clusters_peakvi"

# compute the k-nearest-neighbor graph that is used in both clustering and umap algorithms
sc.pp.neighbors(adata, use_rep=PEAKVI_LATENT_KEY)
# compute the umap
sc.tl.umap(adata, min_dist=0.2)
# cluster the space (we use a lower resolution to get fewer clusters than the default)
sc.tl.leiden(adata, key_added=PEAKVI_CLUSTERS_KEY, resolution=0.2)

model_dir = os.path.join("models", "peakvi_pbmc")
model.save(model_dir, overwrite=True)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `D

Epoch 274/500:  55%|█████▍    | 274/500 [37:27<30:53,  8.20s/it, v_num=1, train_loss=2.01e+8]
Monitored metric reconstruction_loss_validation did not improve in the last 50 records. Best score: 13009.980. Signaling Trainer to stop.


/Users/andreasstampedalgaard/Fagprojekt_Modeling_the_structure_of_DNA/.venv/lib/python3.12/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


: 